In [4]:
pip install numpy pyaudio

  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [14 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build/lib.macosx-11.1-arm64-cpython-312/pyaudio
      copying src/pyaudio/__init__.py -> build/lib.macosx-11.1-arm64-cpython-312/pyaudio
      running build_ext
      building 'pyaudio._portaudio' extension
      creating build/temp.macosx-11.1-arm64-cpython-312/src/pyaudio
      clang -fno-strict-overflow -DNDEBUG -O2 -Wall -fPIC -O2 -isystem /opt/anaconda3/include -arch arm64 -fPIC -O2 -isystem /opt/anaconda3/include -arch arm64 -DMACOS=1 -I/usr/local/include -I/usr/include -I/opt/homebrew/include -I/opt/anaconda3/include/python3.12 -c src/pyaudio/device_api.c -o build

In [2]:
import numpy as np
import pyaudio

def play_tone(freq, waveform_type='sine', duration=0.5, volume=0.5):
  
  # validate parameter input
  if freq < 0:
    raise ValueError("Frequency must be positive.")
  if volume < 0 or volume > 1:
    raise ValueError("Volume must be between 0 and 1.")
  
  # Calculate the number of samples
  sample_rate = 44100 # samples per second
  num_samples = int(sample_rate * duration)

  # Generate the time array
  time_array = np.linspace(0, duration, num_samples, False)
  
  # Generate waveform
  if waveform_type == 'sine':
    waveform = volume * np.sin(freq * 2 * np.pi * time_array)
  elif waveform_type == 'square':
    waveform = volume * np.sign(np.sin(freq * 2 * np.pi * time_array))
  elif waveform_type == 'triangle':
    waveform = volume * (2 * np.abs(2 * (time_array * freq % 1) - 1) - 1)
  elif waveform_type == 'sawtooth':
    waveform = volume * (2 * (time_array * freq % 1) - 1)
  else:
    raise ValueError("Invalid waveform type")

  # Initialize PyAudio
  p = pyaudio.PyAudio()

  # Open the stream
  stream = p.open(format=pyaudio.paFloat32,
      channels=1,
      rate=sample_rate,
      output=True)

  # Play the waveform
  stream.write(waveform.astype(np.float32).tobytes())

  # Close the stream and terminate PyAudio
  stream.stop_stream()
  stream.close()
  p.terminate()

# Example usage:
play_tone(440)
play_tone(440, 'square')
play_tone(440, 'triangle')
play_tone(440, 'sawtooth')


ModuleNotFoundError: No module named 'pyaudio'

CGPT-generated violin synthesizer

In [21]:
import numpy as np
import pyaudio

# Define the fundamental frequency of the sound (A4 note)
fund_freq = 440.0

# Define the amplitudes of the partials (up to the 10th harmonic)
partial_amps = [1.0, 0.67, 0.34, 0.23, 0.1, 0.05, 0.03, 0.02, 0.01, 0.01]

# Define the relative frequencies of the partials
partial_freqs = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]

# Define the duration of the sound
duration = 3.0

# Define the sampling rate and the number of samples
sr = 44100
num_samples = int(sr * duration)

# Generate the waveform
t = np.linspace(0, duration, num_samples, False)  # time array
waveform = np.zeros(num_samples)
for freq, amp in zip(partial_freqs, partial_amps):
    waveform += amp * np.sin(2 * np.pi * freq * fund_freq * t)

# Normalize the waveform
waveform /= np.max(np.abs(waveform))

# Initialize PyAudio
p = pyaudio.PyAudio()

# Open a stream for playing the sound
stream = p.open(format=pyaudio.paFloat32,
                channels=1,
                rate=sr,
                output=True)

# Play the sound
stream.write(waveform.astype(np.float32).tostring())

# Close the stream and terminate PyAudio
stream.stop_stream()
stream.close()
p.terminate()


/var/folders/ml/f5gs6vg976s3bv00yf3q6_gc0000gn/T/ipykernel_2362/3150071783.py:39: DeprecationWarning: tostring() is deprecated. Use tobytes() instead.
  stream.write(waveform.astype(np.float32).tostring())
